# Basketbal Vlaanderen (VBL) — Foutenanalyse & Technische Fouten per Speler

Dit notebook analyseert wedstrijdverslagen van **Basketbal Vlaanderen (VBL / Wisseq)** voor een gekozen club en ploeg.
Het automatiseert de volledige flow:

1. **Club inladen**: Leest de clubgegevens op basis van de club-GUID (bv. `BVBL1037` voor BBC Haantjes Oudenaarde).
2. **Ploeg selecteren**: Lijst alle ploegen van de club op en laat een specifieke ploeg selecteren (bv. `BVBL1037HSE  1` voor HSE A).
3. **Voorbije wedstrijden (Uitslagen)**: Haalt alle gespeelde wedstrijden op van het huidige seizoen.
4. **Wedstrijdverslagen (DWF)**: Doorloopt elke wedstrijd-GUID.
5. **Uitgebreid verslag (Matchdetailpdf)**: Vraagt via de VBL WebService de digitale wedstrijdbladen op:
   - Deelnemers / Spelerslijst (`DwfDeelByWedGuid`)
   - Voortgang / Gebeurtenissen (`DwfVgngByWedGuid`)
6. **Fouten per speler per type**: Brengt naast de naam van elke speler alle gemaakte fouten in kaart, uitgesplitst per type:
   - **Persoonlijke fouten** (`P`, `P1`, `P2`, `P3`)
   - **Technische fouten** (`T`, `T1`, `FL`, `FL2`, `C`, `C1`, `B`, `B1`)
   - **Onsportieve fouten** (`U`, `U1`, `U2`)
   - **Diskwalificerende fouten** (`D`, `D1`, `D2`, `DI`, `DI2`, `GD`)

In [29]:
import requests
import json
import pandas as pd
import matplotlib.pyplot as plt

# VBL API Base URL & Headers (conform Wisseq / VBL DWF)
VBL_BASE_URL = "https://vblcb.wisseq.eu/VBLCB_WebService/data"

# Authenticatie:
# - Voor Senioren (HSE/DSE): GEEN login nodig, de VBL server geeft wedstrijdverslagen publiek vrij ("na").
# - Voor Jeugd (J18, J16, G14...): VBL schermt het speelverloop van minderjarigen af voor niet-ingelogde gebruikers.
#   Omdat jij op vblweb.wisseq.eu w?l bent ingelogd als clublid, kan je jouw sessie-token eenvoudig kopi?ren:
#   1. Open https://vblweb.wisseq.eu in je browser (waar je bent ingelogd)
#   2. Druk op F12 (of rechtermuisknop -> Inspecteren) en klik op het tabblad 'Console'
#   3. Typ: JSON.parse(localStorage.getItem('lsUHeader')) of localStorage.getItem('lsUHeader') en druk op Enter
#   4. Plak die string hieronder tussen de aanhalingstekens (bv. 'Basic bXA6...'):
VBL_AUTH_HEADER = "na"

API_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "authorization": VBL_AUTH_HEADER,
    "ssoBearer": "ssoWQAndre",
    "Content-Type": "application/json; charset=utf-8",
}

print("Setup voltooid. Klaar om VBL data op te halen!")

Setup voltooid. Klaar om VBL data op te halen!


## Stap 1 & 2: Club inladen en ploeg selecteren

We halen alle ploegen van de club op via het `OrgMatchesByGuid` endpoint. We nemen standaard **BBC Haantjes Oudenaarde (`BVBL1037`)** als voorbeeld.

In [43]:
CLUB_GUID = "BVBL1037"  # BBC Haantjes Certifisc Oudenaarde

def get_club_teams(club_guid: str) -> pd.DataFrame:
    """Haalt alle unieke ploegen van een club op uit het wedstrijdprogramma."""
    url = f"{VBL_BASE_URL}/OrgMatchesByGuid?issguid={club_guid}"
    resp = requests.get(url, headers=API_HEADERS, timeout=15)
    resp.raise_for_status()
    matches = resp.json()
    
    teams = {}
    for m in matches:
        if club_guid in m.get("tTGUID", ""):
            teams[m.get("tTGUID")] = m.get("tTNaam")
        if club_guid in m.get("tUGUID", ""):
            teams[m.get("tUGUID")] = m.get("tUNaam")
            
    df_teams = pd.DataFrame([
        {"team_guid": guid, "team_naam": naam}
        for guid, naam in sorted(teams.items())
    ])
    return df_teams

df_teams = get_club_teams(CLUB_GUID)
print(f"Gevonden ploegen voor {CLUB_GUID} ({len(df_teams)} ploegen):")
display(df_teams)

Gevonden ploegen voor BVBL1037 (24 ploegen):


,team_guid,team_naam
0,BVBL1037DSE 1,BBC Haantjes Certifisc Oudenaarde DSE A
1,BVBL1037G08 1,BBC Haantjes Certifisc Oudenaarde G08 A
2,BVBL1037G08 2,BBC Haantjes Certifisc Oudenaarde G08 B
3,BVBL1037G08 3,BBC Haantjes Certifisc Oudenaarde G08 C
4,BVBL1037G10 1,BBC Haantjes Certifisc Oudenaarde G10 A
5,BVBL1037G10 2,BBC Haantjes Certifisc Oudenaarde G10 B
6,BVBL1037G10 3,BBC Haantjes Certifisc Oudenaarde G10 C
7,BVBL1037G12 1,BBC Haantjes Certifisc Oudenaarde G12 A
8,BVBL1037G12 2,BBC Haantjes Certifisc Oudenaarde G12 B
9,BVBL1037G12 3,BBC Haantjes Certifisc Oudenaarde G12 C


### Selecteer de gewenste ploeg
Hieronder kiezen we de ploeg waarvan we de wedstrijden en fouten willen analyseren. Standaard staat dit op **HSE A** (`BVBL1037HSE  1`).

In [39]:
# Kies hier de gewenste team_guid uit de tabel hierboven:
SELECTED_TEAM_GUID = "BVBL1037HSE  1"  # bv. HSE A
SELECTED_TEAM_GUID = "BVBL1037J18  1"

selected_team_name = df_teams.loc[df_teams["team_guid"] == SELECTED_TEAM_GUID, "team_naam"].values[0]
print(f"Geselecteerde ploeg: {selected_team_name} ({SELECTED_TEAM_GUID})")

Geselecteerde ploeg: BBC Haantjes Certifisc Oudenaarde J18 A (BVBL1037J18  1)


## Stap 3: Voorbije wedstrijden (Uitslagen) ophalen

We roepen het `TeamManage` endpoint aan met commando `GETWEDS` om de volledige wedstrijdkalender van deze ploeg op te halen, en filteren op de reeds gespeelde wedstrijden (uitslagen).

In [40]:
def get_team_played_matches(team_guid: str) -> pd.DataFrame:
    """Haalt alle gespeelde wedstrijden (tab uitslagen) op voor een ploeg."""
    payload = {
        "AuthHeader": VBL_AUTH_HEADER,
        "WQVer": "nov2",
        "Command": "GETWEDS",
        "TeamGUID": team_guid,
        "CRUD": "R",
    }
    resp = requests.put(f"{VBL_BASE_URL}/TeamManage", headers=API_HEADERS, json=payload, timeout=15)
    resp.raise_for_status()
    all_matches = resp.json().get("wedList", [])
    
    # Filter op gespeelde wedstrijden (waar een uitslag voor gekend is)
    played = [m for m in all_matches if m.get("uitslag") and str(m.get("uitslag")).strip()]
    df = pd.DataFrame(played)
    
    cols = ["guid", "datumString", "beginTijd", "tTNaam", "tUNaam", "uitslag", "pouleNaam"]
    return df[[c for c in cols if c in df.columns]].rename(columns={
        "guid": "wedguid",
        "datumString": "datum",
        "beginTijd": "uur",
        "tTNaam": "thuisploeg",
        "tUNaam": "uitploeg",
        "pouleNaam": "reeks",
    })

df_played = get_team_played_matches(SELECTED_TEAM_GUID)
print(f"Aantal gespeelde wedstrijden: {len(df_played)}")    
display(df_played)

Aantal gespeelde wedstrijden: 11


,wedguid,datum,uur,thuisploeg,uitploeg,uitslag,reeks
0,BVBL26279100BLAJ18PNCD,22-08-2026,13.00,Joppo Basket Izegem J18 A,BBC Haantjes Certifisc Oudenaarde J18 A,65- 89,Beker van Vlaanderen U18 Poule N
1,BVBL26279170INJ1821BBE,25-10-2026,13.00,BC Lamett Deerlijk-Zwevegem J18 A,BBC Haantjes Certifisc Oudenaarde J18 A,62- 92,U18 Niveau 2 B
2,BVBL26279170INJ1821BCE,04-10-2026,14.00,Joppo Basket Izegem J18 A,BBC Haantjes Certifisc Oudenaarde J18 A,61- 77,U18 Niveau 2 B
3,BVBL26279170INJ1821BDE,27-09-2026,13.00,KBBC Sparta Laarne J18 A,BBC Haantjes Certifisc Oudenaarde J18 A,49- 93,U18 Niveau 2 B
4,BVBL26279170INJ1821BFE,12-09-2026,15.00,Holstra WINGS Wevelgem-Moorsele J18 A,BBC Haantjes Certifisc Oudenaarde J18 A,56- 70,U18 Niveau 2 B
5,BVBL26279100BLAJ18PNDA,29-08-2026,14.15,BBC Haantjes Certifisc Oudenaarde J18 A,Kangoeroes Basket Mechelen J18 A,72- 97,Beker van Vlaanderen U18 Poule N
6,BVBL26279100BLAJ18PNDB,06-09-2026,14.45,BBC Haantjes Certifisc Oudenaarde J18 A,Basket Willebroek J18 A,67- 42,Beker van Vlaanderen U18 Poule N
7,BVBL26279170INJ1821BEA,11-10-2026,14.15,BBC Haantjes Certifisc Oudenaarde J18 A,BBC Helios SanoRice Zottegem J18 A,112- 90,U18 Niveau 2 B
8,BVBL26279170INJ1821BEB,20-09-2026,14.15,BBC Haantjes Certifisc Oudenaarde J18 A,BC Lamett Deerlijk-Zwevegem J18 A,83- 63,U18 Niveau 2 B
9,BVBL26279170INJ1821BED,15-11-2026,14.15,BBC Haantjes Certifisc Oudenaarde J18 A,KBBC Sparta Laarne J18 A,122- 54,U18 Niveau 2 B


## Stap 4 & 5: Wedstrijdverslagen ophalen via DWF (Matchdetailpdf)

Wanneer je op VBL Web op een wedstrijd klikt en het **uitgebreid verslag** (`Matchdetailpdf`) opent, haalt de webapp twee datastromen op:
1. `DwfDeelByWedGuid`: Deelnemerslijst (spelers, rugnummers, coaches) van beide teams (`TtDeel` en `TuDeel`).
2. `DwfVgngByWedGuid`: Chronologische wedstrijdgebeurtenissen (`GebNis`), waarbij `GebType == 30` alle fouten voorstelt.

Onderstaande functie combineert deze twee stromen en normaliseert alle foutcodes (zoals `P`, `P1`, `P2`, `P3`, `(T1)`, `(FL2)`, `C1`, `(B1)`, `DI`, `DI2`).

In [41]:
# Betekenis van alle mogelijke foutcodes in Basketbal Vlaanderen / FIBA DWF
FOUL_EXPLANATIONS = {
    "P": "Persoonlijke fout zonder vrijworpen (balinworp)",
    "P1": "Persoonlijke fout met 1 vrijworp (and-one)",
    "P2": "Persoonlijke fout met 2 vrijworpen (op 2-punter of teamfouten)",
    "P3": "Persoonlijke fout met 3 vrijworpen (op driepunter)",
    "T": "Technische fout speler (1 vrijworp)",
    "T1": "Technische fout speler (1 vrijworp)",
    "FL": "Flopping technische fout (theatrale val, 1 vrijworp)",
    "FL2": "Flopping technische fout (1 vrijworp)",
    "C": "Technische fout coach direct (voor eigen gedrag)",
    "C1": "Technische fout coach direct (voor eigen gedrag)",
    "B": "Technische fout bank / assistent / reservespelers",
    "B1": "Technische fout bank / assistent / reservespelers",
    "U": "Onsportieve fout (2 vrijworpen + balbezit)",
    "U1": "Onsportieve fout met 1 vrijworp + balbezit",
    "U2": "Onsportieve fout met 2 vrijworpen + balbezit",
    "D": "Diskwalificerende fout (onmiddellijke uitsluiting)",
    "D1": "Diskwalificerende fout met 1 vrijworp",
    "D2": "Diskwalificerende fout met 2 vrijworpen",
    "DI": "Diskwalificerende fout (onmiddellijke uitsluiting)",
    "DI2": "Diskwalificerende fout met 2 vrijworpen",
    "GD": "Game Disqualification (uitsluiting na 5 fouten of 2x T/U)",
}

def parse_match_fouls(wedguid: str) -> list:
    """Haalt deelnemers en fouten op voor één wedstrijd-GUID."""
    payload = {
        "AuthHeader": VBL_AUTH_HEADER,
        "WQVer": "nov2",
        "WedGUID": wedguid,
        "CRUD": "R",
    }
    
    try:
        r_deel = requests.put(f"{VBL_BASE_URL}/DwfDeelByWedGuid", headers=API_HEADERS, json=payload, timeout=10)
        r_vgng = requests.put(f"{VBL_BASE_URL}/DwfVgngByWedGuid", headers=API_HEADERS, json=payload, timeout=10)
    except Exception as e:
        print(f"Fout bij ophalen van {wedguid}: {e}")
        return []
        
    if r_deel.status_code != 200 or r_vgng.status_code != 200:
        return []
        
    try:
        data_deel = r_deel.json() if r_deel.text else {}
    except Exception:
        data_deel = {}

    try:
        data_vgng = r_vgng.json() if r_vgng.text else {}
    except Exception:
        data_vgng = {}

    if not isinstance(data_deel, dict):
        data_deel = {}
    if not isinstance(data_vgng, dict):
        data_vgng = {}
        
    # Bouw opzoekwoordenboek van spelers op basis van RelGUID
    players = {}
    for p in (data_deel.get("TtDeel") or []):
        if isinstance(p, dict):
            players[p.get("RelGUID")] = {
                "naam": p.get("Naam"),
                "rugnr": p.get("RugNr"),
                "team_kant": "Thuis",
                "functie": p.get("Functie"),
            }
    for p in (data_deel.get("TuDeel") or []):
        if isinstance(p, dict):
            players[p.get("RelGUID")] = {
                "naam": p.get("Naam"),
                "rugnr": p.get("RugNr"),
                "team_kant": "Uit",
                "functie": p.get("Functie"),
            }
        
    # Doorloop alle gebeurtenissen en filter op fouten (GebType == 30)
    foul_records = []
    geb_list = data_vgng.get("GebNis") or []
    for g in geb_list:
        if isinstance(g, dict) and g.get("GebType") == 30:  # 30 = Fout
            rel = g.get("RelGUID")
            p_info = players.get(rel, {
                "naam": f"Onbekend / Bank ({rel})" if rel else "Bank / Onbekend",
                "rugnr": g.get("RugNr", ""),
                "team_kant": "Thuis" if g.get("TofU") == "T" else "Uit",
                "functie": "S",
            })
            
            raw_code = str(g.get("Text", "")).strip()
            clean_code = raw_code.strip("()")
            
            # Bepaal categorie en omschrijving van de fout
            if clean_code.startswith("P"):
                cat = "Persoonlijk (P)"
            elif clean_code.startswith("T"):
                cat = "Technisch (T)"
            elif clean_code.startswith("FL"):
                cat = "Flopping (FL)"
            elif clean_code.startswith("C"):
                cat = "Technisch Coach (C)"
            elif clean_code.startswith("B"):
                cat = "Technisch Bank (B)"
            elif clean_code.startswith("U"):
                cat = "Onsportief (U)"
            elif clean_code.startswith("D") or clean_code.startswith("GD"):
                cat = "Diskwalificerend (D)"
            else:
                cat = "Overig"
                
            omschr = FOUL_EXPLANATIONS.get(clean_code, FOUL_EXPLANATIONS.get(raw_code, f"Fout ({raw_code})"))
            
            foul_records.append({
                "wedguid": wedguid,
                "RelGUID": rel,
                "Naam": p_info["naam"],
                "RugNr": p_info["rugnr"],
                "TeamKant": p_info["team_kant"],
                "FoutType": clean_code,
                "FoutTypeRaw": raw_code,
                "FoutCategorie": cat,
                "Omschrijving": omschr,
                "Periode": g.get("Periode"),
                "Minuut": g.get("Minuut"),
            })
            
    return foul_records

print("Functie parse_match_fouls gereed.")

Functie parse_match_fouls gereed.


### Test op één wedstrijd & Referentietabel van alle Fouttypes

Hieronder tonen we eerst de **referentietabel met alle officiële fouttypes van Basketbal Vlaanderen**.
Vervolgens testen we op een wedstrijd-GUID (bv. `BVBL26279100LAHSE21ACA`, die zowel Technisch, Bank, Diskwalificerend als Persoonlijk bevat).

In [42]:
# 1. Referentietabel van alle mogelijke foutcodes in Basketbal Vlaanderen (VBL / FIBA)
VBL_FOUL_TYPES = [
    {"Code": "P", "Categorie": "Persoonlijk (P)", "Omschrijving": "Persoonlijke fout zonder vrijworpen (balinworp)"},
    {"Code": "P1", "Categorie": "Persoonlijk (P)", "Omschrijving": "Persoonlijke fout met 1 vrijworp (and-one raak shot)"},
    {"Code": "P2", "Categorie": "Persoonlijk (P)", "Omschrijving": "Persoonlijke fout met 2 vrijworpen (op 2-punter of teamfouten)"},
    {"Code": "P3", "Categorie": "Persoonlijk (P)", "Omschrijving": "Persoonlijke fout met 3 vrijworpen (fout op driepunter)"},
    {"Code": "T / T1", "Categorie": "Technisch (T)", "Omschrijving": "Technische fout op speler (1 vrijworp + balbezit)"},
    {"Code": "FL / FL2", "Categorie": "Flopping (FL)", "Omschrijving": "Flopping technische fout (theatrale val, 1 vrijworp)"},
    {"Code": "C / C1", "Categorie": "Technisch Coach (C)", "Omschrijving": "Technische fout direct op de coach voor eigen gedrag"},
    {"Code": "B / B1", "Categorie": "Technisch Bank (B)", "Omschrijving": "Technische fout op de bank / assistent / reservespelers"},
    {"Code": "U / U1 / U2", "Categorie": "Onsportief (U)", "Omschrijving": "Onsportieve fout (harde fout; 2 vrijworpen + balbezit)"},
    {"Code": "D / DI / DI2", "Categorie": "Diskwalificerend (D)", "Omschrijving": "Diskwalificerende fout (onmiddellijke uitsluiting en zaal verlaten)"},
    {"Code": "GD", "Categorie": "Uitsluiting (GD)", "Omschrijving": "Game Disqualification (uitsluiting na 5 fouten of 2x T/U)"},
]

print("=== REFERENTIETABEL: ALLE MOGELIJKE VBL FOUTTYPES ===")
display(pd.DataFrame(VBL_FOUL_TYPES))

# 2. Testen op een wedstrijd-GUID
# Voorbeelden met diverse fouttypes uit het seizoen:
# - "BVBL26279100LAHSE21ACA" -> Waregem vs Haantjes: bevat T1 (Technisch), B1 (Bank), DI2 (Diskwalificerend), P, P1, P2
# - "BVBL26279100LAHSE21AAJ" -> Haantjes vs Izegem: bevat FL2 (Flopping), T1, P, P1, P2
# - "BVBL26279100BLAHSEPEBD" -> Haantjes vs Knokke: bevat C1 (Coach), P, P1, P2
# - "BVBL26279100LAHSE21AAN" -> Haantjes vs Avanti: bevat DI (Diskwalificerend), P, P2
TEST_WEDGUID = "BVBL26279100LAHSE21ACA"  # Pas gerust aan naar een andere wedguid
TEST_WEDGUID = "BVBL26279170INJ1821BCE"
# TEST_WEDGUID = "BVBL26279170INJ1821BCE"  # Pas gerust aan naar een andere wedguid
test_fouls = parse_match_fouls(TEST_WEDGUID)
df_test = pd.DataFrame(test_fouls)

print(f"\n=== Resultaten voor testwedstrijd {TEST_WEDGUID} ===")
if df_test.empty:
    print(f"⚠️ Geen digitale foutenregistratie (DWF) gevonden voor wedstrijd {TEST_WEDGUID}.")
    print("Mogelijke redenen:")
    print("- Voor deze wedstrijd werd geen digitaal wedstrijdblad (DWF play-by-play) bijgehouden.")
    print("- De wedstrijd is nog niet gespeeld of werd forfait gegeven.")
    print("Kies een andere wedstrijd-GUID uit 'df_played', bv: BVBL26279100LAHSE21ACA")
else:
    print(f"Totaal aantal geregistreerde fouten: {len(df_test)}")
    display(df_test[["TeamKant", "Naam", "RugNr", "FoutType", "FoutCategorie", "Omschrijving", "Periode", "Minuut"]].head(10))
    
    print("\n📊 Aantal fouten per speler per specifiek FOUTTYPE (P, P1, P2, T1, B1, DI2, ...):")
    display(pd.crosstab(
        index=[df_test["TeamKant"], df_test["Naam"], df_test["RugNr"]],
        columns=df_test["FoutType"],
        margins=True,
        margins_name="Totaal"
    ))
    
    print("\n📊 Aantal fouten per speler per FOUTCATEGORIE (Persoonlijk, Technisch, Bank, Diskwalificerend, ...):")
    display(pd.crosstab(
        index=[df_test["TeamKant"], df_test["Naam"], df_test["RugNr"]],
        columns=df_test["FoutCategorie"],
        margins=True,
        margins_name="Totaal"
    ))

=== REFERENTIETABEL: ALLE MOGELIJKE VBL FOUTTYPES ===


,Code,Categorie,Omschrijving
0,P,Persoonlijk (P),Persoonlijke fout zonder vrijworpen (balinworp)
1,P1,Persoonlijk (P),Persoonlijke fout met 1 vrijworp (and-one raak...
2,P2,Persoonlijk (P),Persoonlijke fout met 2 vrijworpen (op 2-punte...
3,P3,Persoonlijk (P),Persoonlijke fout met 3 vrijworpen (fout op dr...
4,T / T1,Technisch (T),Technische fout op speler (1 vrijworp + balbezit)
5,FL / FL2,Flopping (FL),"Flopping technische fout (theatrale val, 1 vri..."
6,C / C1,Technisch Coach (C),Technische fout direct op de coach voor eigen ...
7,B / B1,Technisch Bank (B),Technische fout op de bank / assistent / reser...
8,U / U1 / U2,Onsportief (U),Onsportieve fout (harde fout; 2 vrijworpen + b...
9,D / DI / DI2,Diskwalificerend (D),Diskwalificerende fout (onmiddellijke uitsluit...



=== Resultaten voor testwedstrijd BVBL26279170INJ1821BCE ===
⚠️ Geen digitale foutenregistratie (DWF) gevonden voor wedstrijd BVBL26279170INJ1821BCE.
Mogelijke redenen:
- Voor deze wedstrijd werd geen digitaal wedstrijdblad (DWF play-by-play) bijgehouden.
- De wedstrijd is nog niet gespeeld of werd forfait gegeven.
Kies een andere wedstrijd-GUID uit 'df_played', bv: BVBL26279100LAHSE21ACA


## Stap 6: Alle voorbije wedstrijden doorlopen en fouten per speler aggregeren

Nu doorlopen we **alle gespeelde wedstrijden** van de geselecteerde ploeg. We verzamelen alle fouten en bouwen een overzichtstabel per speler van onze club.

In [36]:
all_season_fouls = []

print(f"Bezig met verwerken van {len(df_played)} wedstrijden voor {selected_team_name}...")
for idx, row in df_played.iterrows():
    w_guid = row["wedguid"]
    w_datum = row["datum"]
    w_thuis = row["thuisploeg"]
    w_uit = row["uitploeg"]
    
    # Bepaal of onze ploeg thuis of uit speelde
    is_own_home = (SELECTED_TEAM_GUID in row.get("tTGUID", "")) or ("Haantjes" in w_thuis)
    own_side = "Thuis" if is_own_home else "Uit"
    
    fouls = parse_match_fouls(w_guid)
    for f in fouls:
        f["datum"] = w_datum
        f["match_label"] = f"{w_thuis} vs {w_uit}"
        f["is_eigen_speler"] = (f["TeamKant"] == own_side)
        all_season_fouls.append(f)

df_all_fouls = pd.DataFrame(all_season_fouls)
print(f"Klaar! Totaal geregistreerde fouten: {len(df_all_fouls)}")

Bezig met verwerken van 14 wedstrijden voor BBC Haantjes Certifisc Oudenaarde HSE A...
Klaar! Totaal geregistreerde fouten: 560


### Volledig Overzicht: Fouten per speler van de eigen ploeg

We filteren op de spelers van de eigen club en maken twee overzichtelijke tabellen:
1. **Per specifiek FoutType** (`P`, `P1`, `P2`, `P3`, `T1`, `FL2`, `C1`, `B1`, `DI`, `DI2`, ...)
2. **Samengevat per Hoofdcategorie** (`Persoonlijk`, `Technisch`, `Flopping`, `Coach/Bank`, `Diskwalificerend`, `TOTAAL`)

In [37]:
# Filter enkel op spelers van onze eigen ploeg
df_own_fouls = df_all_fouls[df_all_fouls["is_eigen_speler"]].copy()

if df_own_fouls.empty:
    print("Geen fouten gevonden voor de eigen ploeg.")
else:
    # 1. Pivot tabel per speler x individueel fouttype
    pivot_player_fouls = pd.crosstab(
        index=df_own_fouls["Naam"],
        columns=df_own_fouls["FoutType"],
        margins=False
    )
    
    # Bereken totalen per hoofdcategorie
    p_cols = [c for c in pivot_player_fouls.columns if c.startswith("P")]
    t_cols = [c for c in pivot_player_fouls.columns if c.startswith("T") or c.startswith("C") or c.startswith("B")]
    fl_cols = [c for c in pivot_player_fouls.columns if c.startswith("FL")]
    u_cols = [c for c in pivot_player_fouls.columns if c.startswith("U")]
    d_cols = [c for c in pivot_player_fouls.columns if c.startswith("D")]
    
    summary = pivot_player_fouls.copy()
    summary["Totaal_P (Persoonlijk)"] = summary[p_cols].sum(axis=1) if p_cols else 0
    summary["Totaal_T (Technisch)"] = summary[t_cols].sum(axis=1) if t_cols else 0
    summary["Totaal_FL (Flopping)"] = summary[fl_cols].sum(axis=1) if fl_cols else 0
    summary["Totaal_U (Onsportief)"] = summary[u_cols].sum(axis=1) if u_cols else 0
    summary["Totaal_D (Diskwalificerend)"] = summary[d_cols].sum(axis=1) if d_cols else 0
    summary["TOTAAL"] = (
        summary["Totaal_P (Persoonlijk)"] +
        summary["Totaal_T (Technisch)"] +
        summary["Totaal_FL (Flopping)"] +
        summary["Totaal_U (Onsportief)"] +
        summary["Totaal_D (Diskwalificerend)"]
    )
    
    # Sorteer op Technisch + Flopping + Diskwalificerend eerst (de opvallendste fouten), dan Totaal
    sort_key = summary["Totaal_T (Technisch)"] + summary["Totaal_FL (Flopping)"] + summary["Totaal_D (Diskwalificerend)"]
    summary = summary.assign(_sort=sort_key).sort_values(by=["_sort", "TOTAAL"], ascending=False).drop(columns=["_sort"])
    
    print(f"📊 1. Volledig overzicht per specifiek FoutType voor {selected_team_name}:")
    display(summary)
    
    print(f"\n📊 2. Overzicht samengevat per FoutCategorie voor {selected_team_name}:")
    pivot_cat = pd.crosstab(
        index=df_own_fouls["Naam"],
        columns=df_own_fouls["FoutCategorie"],
        margins=True,
        margins_name="TOTAAL"
    )
    display(pivot_cat)

📊 1. Volledig overzicht per specifiek FoutType voor BBC Haantjes Certifisc Oudenaarde HSE A:


FoutType,DI,DI2,FL2,P,P1,P2,P3,T1,Totaal_P (Persoonlijk),Totaal_T (Technisch),Totaal_FL (Flopping),Totaal_U (Onsportief),Totaal_D (Diskwalificerend),TOTAAL
Naam,,,,,,,,,,,,,,
Robin Spaens,1,1,2,9,2,16,1,0,28,0,2,0,2,32
Jakob Marcou,0,0,0,9,3,15,0,2,27,2,0,0,0,29
Cédric Watteyn,0,1,0,20,4,13,0,0,37,0,0,0,1,38
Marco Dick,0,0,0,8,1,22,0,1,31,1,0,0,0,32
Ukeme Essien,0,0,0,11,2,14,0,1,27,1,0,0,0,28
Quinten Derous,0,1,0,1,3,10,0,0,14,0,0,0,1,15
Andreas Lippens,0,0,0,18,0,18,0,0,36,0,0,0,0,36
Maxime Bleys,0,0,0,15,4,7,0,0,26,0,0,0,0,26
Niels De Lepeleire,0,0,0,16,1,6,1,0,24,0,0,0,0,24



📊 2. Overzicht samengevat per FoutCategorie voor BBC Haantjes Certifisc Oudenaarde HSE A:


FoutCategorie,Diskwalificerend (D),Flopping (FL),Persoonlijk (P),Technisch (T),TOTAAL
Naam,,,,,
Andreas Lippens,0,0,36,0,36
Bogdan Verstraete,0,0,1,0,1
Cédric Watteyn,1,0,37,0,38
Jakob Marcou,0,0,27,2,29
Marco Dick,0,0,31,1,32
Maxime Bleys,0,0,26,0,26
Milan De Brabander,0,0,3,0,3
Niels De Lepeleire,0,0,24,0,24
Quinten Derous,1,0,14,0,15


### Detail van Speciale Fouten (Technisch, Flopping, Diskwalificerend, Onsportief)
Hier lichten we specifiek alle niet-persoonlijke fouten uit, inclusief de datum, wedstrijd, periode en minuut waarop ze zijn opgelopen.

In [38]:
speciale_fouten = df_own_fouls[df_own_fouls["FoutCategorie"] != "Persoonlijk (P)"].sort_values(by=["FoutCategorie", "Naam", "datum"])

if speciale_fouten.empty:
    print("Geen technische, flopping, onsportieve of diskwalificerende fouten geregistreerd voor deze ploeg! Fair play!")
else:
    print(f"Aantal speciale fouten (Technisch / Flopping / Diskwalificerend / Onsportief): {len(speciale_fouten)}")
    display(speciale_fouten[["datum", "Naam", "RugNr", "FoutType", "FoutCategorie", "Omschrijving", "Periode", "Minuut", "match_label"]])

Aantal speciale fouten (Technisch / Flopping / Diskwalificerend / Onsportief): 10


,datum,Naam,RugNr,FoutType,FoutCategorie,Omschrijving,Periode,Minuut,match_label
380,11-11-2026,Cédric Watteyn,4,DI2,Diskwalificerend (D),Diskwalificerende fout met 2 vrijworpen,1,7,G&V Breakpoint Basket Waregem HSE B vs BBC Haa...
420,11-11-2026,Quinten Derous,11,DI2,Diskwalificerend (D),Diskwalificerende fout met 2 vrijworpen,4,1,G&V Breakpoint Basket Waregem HSE B vs BBC Haa...
267,15-11-2026,Robin Spaens,5,DI,Diskwalificerend (D),Diskwalificerende fout (onmiddellijke uitsluit...,2,3,BBC Haantjes Certifisc Oudenaarde HSE A vs Ava...
172,25-10-2026,Robin Spaens,5,DI2,Diskwalificerend (D),Diskwalificerende fout met 2 vrijworpen,4,8,BBC Haantjes Certifisc Oudenaarde HSE A vs Bas...
544,11-10-2026,Robin Spaens,5,FL2,Flopping (FL),Flopping technische fout (1 vrijworp),4,2,ABO LDP Donza HSE B vs BBC Haantjes Certifisc ...
553,11-10-2026,Robin Spaens,5,FL2,Flopping (FL),Flopping technische fout (1 vrijworp),4,4,ABO LDP Donza HSE B vs BBC Haantjes Certifisc ...
519,11-10-2026,Jakob Marcou,7,T1,Technisch (T),Technische fout speler (1 vrijworp),2,3,ABO LDP Donza HSE B vs BBC Haantjes Certifisc ...
408,11-11-2026,Jakob Marcou,7,T1,Technisch (T),Technische fout speler (1 vrijworp),3,5,G&V Breakpoint Basket Waregem HSE B vs BBC Haa...
549,11-10-2026,Marco Dick,10,T1,Technisch (T),Technische fout speler (1 vrijworp),4,2,ABO LDP Donza HSE B vs BBC Haantjes Certifisc ...
231,04-10-2026,Ukeme Essien,14,T1,Technisch (T),Technische fout speler (1 vrijworp),2,10,BBC Haantjes Certifisc Oudenaarde HSE A vs Jop...


## Stap 7: Visualisatie

We visualiseren de verdeling van fouten over de spelers en wedstrijden.

In [ ]:
if not df_own_fouls.empty and not summary.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    top_players = summary.head(12)
    
    # Gestapelde staafdiagram per categorie
    bars_p = ax.bar(top_players.index, top_players["Totaal_P (Persoonlijk)"], label="Persoonlijk (P)", color="#1f77b4")
    bottom_u = top_players["Totaal_P (Persoonlijk)"]
    bars_u = ax.bar(top_players.index, top_players["Totaal_U (Onsportief)"], bottom=bottom_u, label="Onsportief (U)", color="#ff7f0e")
    bottom_t = bottom_u + top_players["Totaal_U (Onsportief)"]
    bars_t = ax.bar(top_players.index, top_players["Totaal_T (Technisch)"] + top_players["Totaal_FL (Flopping)"], bottom=bottom_t, label="Technisch & Flopping", color="#d62728")
    
    ax.set_title(f"Foutenverdeling per speler — {selected_team_name}", fontsize=14, fontweight="bold", pad=15)
    ax.set_ylabel("Aantal fouten", fontsize=12)
    ax.set_xlabel("Speler", fontsize=12)
    plt.xticks(rotation=45, ha="right")
    ax.legend()
    plt.tight_layout()
    plt.show()

## Stap 8: Digitaal Wedstrijdblad (Matchdetailpdf) inladen en visualiseren

### Waarom opent de directe link `https://vblweb.wisseq.eu/Home/Matchdetailpdf` niet?
De pagina `/Home/Matchdetailpdf` op de VBL-website is **geen gewone REST API of PDF-link**. De pagina verwacht dat de data al in de **`sessionStorage` van de browser** zit (geplaatst nadat je op de matchpagina op *"Maak (uitgebreid) verslag"* hebt geklikt). Als je er rechtstreeks naartoe surft, is die opslag leeg en toont de pagina niets.

Bovendien weigert de knop op de VBL-website voor wedstrijden waar geen live digitaal wedstrijdverloop voor werd bijgehouden (zoals in veel jeugdreeksen).

### Oplossing in Python
Omdat we rechtstreeks communiceren met de VBL WebService (`MatchByWedGuid`, `DwfDeelByWedGuid`, `DwfVgngByWedGuid` en `DwfOffiByWedGuid`), kunnen we voor elke wedstrijd met DWF-data het **exacte offici?le digitale wedstrijdblad** opbouwen en weergeven:

In [ ]:
from IPython.display import display, HTML

def toon_uitgebreid_verslag(wedguid: str, save_html_file: bool = False):
    """Bouwt het offici?le VBL wedstrijdblad (zoals op Matchdetailpdf) op in HTML."""
    # 1. Match info
    r_m = requests.get(f"{VBL_BASE_URL}/MatchByWedGuid?issguid={wedguid}", headers=API_HEADERS, timeout=10)
    if r_m.status_code != 200 or not r_m.text:
        print(f"Wedstrijd {wedguid} niet gevonden.")
        return
    m = r_m.json()[0]

    # 2. Deelnemers, Voortgang en Officials
    p_load = {"AuthHeader": VBL_AUTH_HEADER, "WQVer": "nov2", "WedGUID": wedguid, "CRUD": "R"}
    r_deel = requests.put(f"{VBL_BASE_URL}/DwfDeelByWedGuid", headers=API_HEADERS, json=p_load, timeout=10)
    r_vgng = requests.put(f"{VBL_BASE_URL}/DwfVgngByWedGuid", headers=API_HEADERS, json=p_load, timeout=10)
    r_offi = requests.put(f"{VBL_BASE_URL}/DwfOffiByWedGuid", headers=API_HEADERS, json=p_load, timeout=10)

    d_deel = r_deel.json() if r_deel.status_code == 200 and r_deel.text else {}
    d_vgng = r_vgng.json() if r_vgng.status_code == 200 and r_vgng.text else {}
    d_offi = r_offi.json() if r_offi.status_code == 200 and r_offi.text else {}

    if not d_vgng or not d_vgng.get("GebNis"):
        print(f"?? Geen DWF wedstrijdverloop beschikbaar voor {wedguid}.")
        print("Voor deze wedstrijd werd geen live digitaal wedstrijdblad gebruikt (gebruikelijk bij lagere jeugd).")
        return

    tt_players = {p["RelGUID"]: p for p in (d_deel.get("TtDeel") or []) if isinstance(p, dict)}
    tu_players = {p["RelGUID"]: p for p in (d_deel.get("TuDeel") or []) if isinstance(p, dict)}

    for p in list(tt_players.values()) + list(tu_players.values()):
        p["score"] = 0
        p["sFouten"] = []

    period_totals = {
        1: {"T": 0, "U": 0, "T_f": 0, "U_f": 0, "T_to": 0, "U_to": 0},
        2: {"T": 0, "U": 0, "T_f": 0, "U_f": 0, "T_to": 0, "U_to": 0},
        3: {"T": 0, "U": 0, "T_f": 0, "U_f": 0, "T_to": 0, "U_to": 0},
        4: {"T": 0, "U": 0, "T_f": 0, "U_f": 0, "T_to": 0, "U_to": 0},
    }

    for g in (d_vgng.get("GebNis") or []):
        if not isinstance(g, dict): continue
        g_type = g.get("GebType")
        rel = g.get("RelGUID")
        tofu = g.get("TofU")
        per = g.get("Periode", 1)
        if per == 99: continue
        if per not in period_totals:
            period_totals[per] = {"T": 0, "U": 0, "T_f": 0, "U_f": 0, "T_to": 0, "U_to": 0}

        if g_type == 10:  # Score
            txt = str(g.get("Text", ""))
            pts = int(txt[0]) if txt and txt[0].isdigit() else 0
            if tofu == "T":
                if rel in tt_players: tt_players[rel]["score"] += pts
                period_totals[per]["T"] += pts
            elif tofu == "U":
                if rel in tu_players: tu_players[rel]["score"] += pts
                period_totals[per]["U"] += pts
        elif g_type == 20:  # Timeout
            if tofu == "T": period_totals[per]["T_to"] += 1
            elif tofu == "U": period_totals[per]["U_to"] += 1
        elif g_type == 30:  # Fout
            f_code = str(g.get("Text", "")).strip("()")
            if tofu == "T":
                if rel in tt_players: tt_players[rel]["sFouten"].append(f_code)
                period_totals[per]["T_f"] += 1
            elif tofu == "U":
                if rel in tu_players: tu_players[rel]["sFouten"].append(f_code)
                period_totals[per]["U_f"] += 1

    period_rows = ""
    for per in sorted(period_totals.keys()):
        p_data = period_totals[per]
        period_rows += f"""<tr style='border-bottom:1px solid #ddd;'>
            <td style='font-weight:bold; padding:4px;'>Q{per}</td>
            <td style='text-align:center;'>{p_data['T']} - {p_data['U']}</td>
            <td style='text-align:center;'>{p_data['T_f']}</td>
            <td style='text-align:center;'>{p_data['U_f']}</td>
            <td style='text-align:center;'>{p_data['T_to']}</td>
            <td style='text-align:center;'>{p_data['U_to']}</td>
        </tr>"""

    def render_team_rows(players_dict):
        rows = ""
        for p in players_dict.values():
            if p.get("Functie") != "S": continue
            f = p["sFouten"] + [""] * (5 - len(p["sFouten"]))
            gd = "GD" if len(p["sFouten"]) >= 5 else ""
            r1 = "?" if p.get("Rol1e5") == "Y" else ""
            
            f_cells = ""
            for code in f[:5]:
                is_special = any(code.startswith(k) for k in ["T", "U", "D", "FL", "C", "B"])
                bg = "#ffcdd2" if is_special else ("#fffde7" if code else "#fafafa")
                f_cells += f"<td style='text-align:center; background:{bg}; font-weight:bold; padding:4px; border:1px solid #e0e0e0;'>{code}</td>"
                
            rows += f"""<tr style='border-bottom:1px solid #eee;'>
                <td style='font-weight:bold; text-align:center; padding:4px;'>{p.get('RugNr', '')}</td>
                <td style='padding:4px;'>{p.get('Naam', '')}</td>
                <td style='text-align:center; color:#ff9800;'>{r1}</td>
                {f_cells}
                <td style='text-align:center; color:red; font-weight:bold;'>{gd}</td>
                <td style='text-align:center; font-weight:bold; background:#e8f5e9;'>{p['score']}</td>
            </tr>"""
        return rows

    off_items = "".join([
        f"<li><b>{o.get('Functie', 'Official')}:</b> {o.get('Naam', '')}</li>"
        for o in (d_offi.get("Offs") or []) if isinstance(o, dict) and o.get("Naam")
    ])

    html = f"""
    <div style='font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,sans-serif; max-width:880px; margin:1rem auto; padding:1.5rem; border:1px solid #cdd5df; border-radius:10px; background:white; box-shadow:0 3px 10px rgba(0,0,0,0.08);'>
        <div style='display:flex; justify-content:space-between; align-items:center; border-bottom:2px solid #10243e; padding-bottom:0.75rem; margin-bottom:1rem;'>
            <div>
                <h2 style='margin:0; color:#10243e; font-size:1.3rem;'>?? {m['teamThuisNaam']} vs {m['teamUitNaam']}</h2>
                <div style='color:#666; font-size:0.85rem; margin-top:4px;'>?? {m['pouleNaam']} ? ?? {m['datumString']} ? ?? {m.get('accommodatieDoc', {}).get('naam', '')}</div>
            </div>
            <div style='text-align:right;'>
                <div style='font-size:1.8rem; font-weight:bold; color:#d62728;'>{m.get('uitslag', '')}</div>
                <div style='font-size:0.75rem; color:#888;'>WedID: {m.get('wedID', '')}</div>
            </div>
        </div>

        <div style='display:flex; gap:1.5rem; margin-bottom:1.5rem;'>
            <div style='flex:1;'>
                <h4 style='margin:0 0 0.4rem 0; color:#10243e; font-size:0.95rem;'>?? Kwartstanden</h4>
                <table style='width:100%; border-collapse:collapse; font-size:0.82rem;'>
                    <thead><tr style='background:#f0f3f7;'><th style='padding:4px;'>Q</th><th>Score</th><th>F (Thuis)</th><th>F (Uit)</th><th>TO (T)</th><th>TO (U)</th></tr></thead>
                    <tbody>{period_rows}</tbody>
                </table>
            </div>
            <div style='flex:1;'>
                <h4 style='margin:0 0 0.4rem 0; color:#10243e; font-size:0.95rem;'>????? Officials & Tafelaars</h4>
                <ul style='font-size:0.82rem; line-height:1.6; margin:0; padding-left:1.2rem;'>{off_items}</ul>
            </div>
        </div>

        <h4 style='color:#10243e; margin:0.8rem 0 0.3rem 0; font-size:1rem;'>?? Thuisploeg: {m['teamThuisNaam']}</h4>
        <table style='width:100%; border-collapse:collapse; margin-bottom:1.2rem; font-size:0.85rem;'>
            <thead><tr style='background:#10243e; color:white;'><th style='width:35px;'>Nr</th><th style='text-align:left; padding:4px;'>Speler</th><th style='width:30px;'>1e5</th><th style='width:40px;'>F1</th><th style='width:40px;'>F2</th><th style='width:40px;'>F3</th><th style='width:40px;'>F4</th><th style='width:40px;'>F5</th><th style='width:40px;'>Uitsl</th><th style='width:50px;'>Score</th></tr></thead>
            <tbody>{render_team_rows(tt_players)}</tbody>
        </table>

        <h4 style='color:#10243e; margin:0.8rem 0 0.3rem 0; font-size:1rem;'>?? Uitploeg: {m['teamUitNaam']}</h4>
        <table style='width:100%; border-collapse:collapse; margin-bottom:0.5rem; font-size:0.85rem;'>
            <thead><tr style='background:#10243e; color:white;'><th style='width:35px;'>Nr</th><th style='text-align:left; padding:4px;'>Speler</th><th style='width:30px;'>1e5</th><th style='width:40px;'>F1</th><th style='width:40px;'>F2</th><th style='width:40px;'>F3</th><th style='width:40px;'>F4</th><th style='width:40px;'>F5</th><th style='width:40px;'>Uitsl</th><th style='width:50px;'>Score</th></tr></thead>
            <tbody>{render_team_rows(tu_players)}</tbody>
        </table>
    </div>
    """

    display(HTML(html))
    if save_html_file:
        out_filename = f"wedstrijdverslag_{wedguid}.html"
        with open(out_filename, "w", encoding="utf-8") as f:
            f.write(html)
        print(f"Verslag opgeslagen als '{out_filename}'.")

# Voorbeeld: toont het digitale wedstrijdblad van Waregem vs Haantjes HSE A (met fouten en kwartstanden):
toon_uitgebreid_verslag("BVBL26279100LAHSE21ACA")

## Samenvatting

### Q&A
- **Waarom toont een bepaalde wedstrijd soms 0 fouten?**
  In sommige wedstrijden (bv. lagere jeugd of wanneer er een papieren wedstrijdblad werd gebruikt in plaats van de live digitale DWF) zijn er geen `GebNis`-events opgeslagen in de VBL WebService.
- **Welke fouttypes worden herkend?**
  Alle officiële FIBA/VBL fouttypes: `P`, `P1`, `P2`, `P3`, `T1`, `FL2` (flopping), `C1` (coach), `B1` (bank), `U1`, `U2` (onsportief), `DI`, `DI2`, `D1`, `D2` (diskwalificerend) en `GD` (uitsluiting).

### Data Analysis Key Findings
- De API levert real-time toegang tot het officiële wedstrijdblad zonder dat authenticatie vereist is (`authorization: 'na'`, `ssoBearer: 'ssoWQAndre'`).
- Zowel persoonlijke als technische, flopping, coach, bank en diskwalificerende fouten worden geregistreerd per speler, inclusief periode en minuut.